In [1]:
# STEP 1: Import required libraries

import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

# Display settings for easier inspection
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [2]:

# STEP 2: Set up project paths

PROJECT_ROOT = Path(r"C:\Users\nijag\Desktop\Anjaliiiii")

RAW_EMAIL_DIR = PROJECT_ROOT / "data" / "raw" / "email"
PROCESSED_EMAIL_DIR = PROJECT_ROOT / "data" / "processed" / "email"
STANDARDIZED_DIR = PROCESSED_EMAIL_DIR / "standardized"

# Create output folders if they do not already exist
PROCESSED_EMAIL_DIR.mkdir(parents=True, exist_ok=True)
STANDARDIZED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("Raw email folder exists:", RAW_EMAIL_DIR.exists())
print("Processed email folder exists:", PROCESSED_EMAIL_DIR.exists())
print("Standardized folder exists:", STANDARDIZED_DIR.exists())

Project root: C:\Users\nijag\Desktop\Anjaliiiii
Project root exists: True
Raw email folder exists: True
Processed email folder exists: True
Standardized folder exists: True


In [3]:

# STEP 3: Check that all required dataset files exist

dataset_files = {
    "CEAS_08": "CEAS_08.csv",
    "Enron": "Enron.csv",
    "Ling": "Ling.csv",
    "Nazario": "Nazario.csv",
    "Nigerian_Fraud": "Nigerian_Fraud.csv",
    "phishing_email": "phishing_email.csv",
    "SpamAssasin": "SpamAssasin.csv"
}

print("Checking required email dataset files...\n")

for dataset_name, filename in dataset_files.items():
    file_path = RAW_EMAIL_DIR / filename

    if file_path.exists():
        print(f"FOUND: {filename}")
    else:
        print(f"MISSING: {filename}")

all_files_exist = all(
    (RAW_EMAIL_DIR / filename).exists()
    for filename in dataset_files.values()
)

print("\nAll seven dataset files are available:", all_files_exist)

assert all_files_exist, (
    "One or more dataset files are missing. "
    "Check the raw email folder path and filenames."
)

Checking required email dataset files...

FOUND: CEAS_08.csv
FOUND: Enron.csv
FOUND: Ling.csv
FOUND: Nazario.csv
FOUND: Nigerian_Fraud.csv
FOUND: phishing_email.csv
FOUND: SpamAssasin.csv

All seven dataset files are available: True


In [4]:

# STEP 4: Load all seven email datasets

datasets = {}

for dataset_name, filename in dataset_files.items():
    file_path = RAW_EMAIL_DIR / filename

    datasets[dataset_name] = pd.read_csv(file_path)

    print(f"\nDataset: {dataset_name}")
    print("Shape:", datasets[dataset_name].shape)
    print("Columns:", datasets[dataset_name].columns.tolist())

print("\nAll seven datasets loaded successfully!")


Dataset: CEAS_08
Shape: (39154, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']

Dataset: Enron
Shape: (29767, 3)
Columns: ['subject', 'body', 'label']

Dataset: Ling
Shape: (2859, 3)
Columns: ['subject', 'body', 'label']

Dataset: Nazario
Shape: (1565, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']

Dataset: Nigerian_Fraud
Shape: (3332, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']

Dataset: phishing_email
Shape: (82486, 2)
Columns: ['text_combined', 'label']

Dataset: SpamAssasin
Shape: (5809, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']

All seven datasets loaded successfully!


In [5]:

# STEP 5: Create an initial dataset summary

dataset_summary = []

for dataset_name, df in datasets.items():
    dataset_summary.append({
        "dataset": dataset_name,
        "rows": df.shape[0],
        "columns_count": df.shape[1],
        "column_names": ", ".join(df.columns.astype(str))
    })

dataset_summary = pd.DataFrame(dataset_summary)

print("Initial Dataset Summary:")
display(dataset_summary)

# Save the summary for project documentation
summary_path = PROCESSED_EMAIL_DIR / "dataset_initial_summary.csv"
dataset_summary.to_csv(summary_path, index=False)

print("\nDataset summary saved to:", summary_path)

Initial Dataset Summary:


,dataset,rows,columns_count,column_names
0,CEAS_08,39154,7,"sender, receiver, date, subject, body, label, urls"
1,Enron,29767,3,"subject, body, label"
2,Ling,2859,3,"subject, body, label"
3,Nazario,1565,7,"sender, receiver, date, subject, body, urls, label"
4,Nigerian_Fraud,3332,7,"sender, receiver, date, subject, body, urls, label"
5,phishing_email,82486,2,"text_combined, label"
6,SpamAssasin,5809,7,"sender, receiver, date, subject, body, label, urls"



Dataset summary saved to: C:\Users\nijag\Desktop\Anjaliiiii\data\processed\email\dataset_initial_summary.csv


In [6]:

# STEP 6: Check labels and duplicates in each dataset

label_summary = []

for dataset_name, df in datasets.items():
    print(f"\nDataset: {dataset_name}")

    if "label" in df.columns:
        print("Label distribution:")
        display(df["label"].value_counts(dropna=False).sort_index().to_frame("count"))

        label_0_count = (df["label"] == 0).sum()
        label_1_count = (df["label"] == 1).sum()
        missing_labels = df["label"].isna().sum()
    else:
        label_0_count = 0
        label_1_count = 0
        missing_labels = len(df)

    duplicate_rows = df.duplicated().sum()

    label_summary.append({
        "dataset": dataset_name,
        "label_0_count": int(label_0_count),
        "label_1_count": int(label_1_count),
        "missing_labels": int(missing_labels),
        "duplicate_rows": int(duplicate_rows)
    })

label_summary = pd.DataFrame(label_summary)

print("\nSummary of labels and duplicates:")
display(label_summary)

# Save the results
label_summary_path = (
    PROCESSED_EMAIL_DIR / "label_and_duplicate_summary.csv"
)
label_summary.to_csv(label_summary_path, index=False)

print("\nLabel and duplicate summary saved to:", label_summary_path)


Dataset: CEAS_08
Label distribution:


,count
label,
0,17312
1,21842



Dataset: Enron
Label distribution:


,count
label,
0,15791
1,13976



Dataset: Ling
Label distribution:


...


Dataset: SpamAssasin
Label distribution:


,count
label,
0,4091
1,1718



Summary of labels and duplicates:


,dataset,label_0_count,label_1_count,missing_labels,duplicate_rows
0,CEAS_08,17312,21842,0,0
1,Enron,15791,13976,0,0
2,Ling,2401,458,0,0
3,Nazario,0,1565,0,0
4,Nigerian_Fraud,0,3332,0,0
5,phishing_email,39595,42891,0,408
6,SpamAssasin,4091,1718,0,0



Label and duplicate summary saved to: C:\Users\nijag\Desktop\Anjaliiiii\data\processed\email\label_and_duplicate_summary.csv


In [7]:

# STEP 7: Standardize all datasets

common_columns = [
    "subject", "body", "sender", "receiver",
    "date", "urls", "label", "source"
]

standardized_datasets = {}

for dataset_name, df in datasets.items():

    # Create a new DataFrame with the original row index
    standardized = pd.DataFrame(index=df.index)

    # Copy available columns and create missing columns when necessary
    for column in [
        "subject", "body", "sender", "receiver",
        "date", "urls", "label"
    ]:
        if column in df.columns:
            standardized[column] = df[column]
        else:
            standardized[column] = pd.NA

    # Handle the dataset that stores subject and body together
    if "text_combined" in df.columns:
        standardized["body"] = df["text_combined"]
        standardized["subject"] = pd.NA

    # Record the original dataset name
    standardized["source"] = dataset_name

    # Ensure the same column order for every dataset
    standardized = standardized[common_columns]

    standardized_datasets[dataset_name] = standardized

    # Save each standardized dataset
    output_path = STANDARDIZED_DIR / f"{dataset_name}_standardized.csv"
    standardized.to_csv(output_path, index=False)

    print(f"\nDataset: {dataset_name}")
    print("Standardized shape:", standardized.shape)
    print("Columns:", standardized.columns.tolist())

print("\nAll seven datasets standardized successfully!")
print("Standardized files saved to:", STANDARDIZED_DIR)


Dataset: CEAS_08
Standardized shape: (39154, 8)
Columns: ['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

Dataset: Enron
Standardized shape: (29767, 8)
Columns: ['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

Dataset: Ling
Standardized shape: (2859, 8)
Columns: ['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

Dataset: Nazario
Standardized shape: (1565, 8)
Columns: ['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

Dataset: Nigerian_Fraud
Standardized shape: (3332, 8)
Columns: ['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

Dataset: phishing_email
Standardized shape: (82486, 8)
Columns: ['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

Dataset: SpamAssasin
Standardized shape: (5809, 8)
Columns: ['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

All seven datasets standardized successfully!

In [ ]:
## Step 8: Combine the Standardized Datasets

**Objective:** Merge the seven standardized datasets into one combined email dataset.

**Why:** A single dataset makes it easier to perform consistent cleaning and prepare the emails for feature extraction and machine learning.

**Expected result:** 164,972 rows and 8 columns before duplicate removal.


In [8]:

# STEP 8: Combine all standardized datasets

combined_email = pd.concat(
    standardized_datasets.values(),
    ignore_index=True
)

print("All datasets combined successfully!")
print("Combined dataset shape:", combined_email.shape)

print("\nColumns:")
print(combined_email.columns.tolist())

print("\nRows contributed by each dataset:")
display(combined_email["source"].value_counts().to_frame("row_count"))

print("\nFirst five rows:")
display(combined_email.head())

# Save a copy before duplicate removal
COMBINED_DATA_PATH = (
    PROCESSED_EMAIL_DIR / "email_combined_before_cleaning.csv"
)

combined_email.to_csv(COMBINED_DATA_PATH, index=False)

print("\nCombined dataset saved to:", COMBINED_DATA_PATH)

All datasets combined successfully!
Combined dataset shape: (164972, 8)

Columns:
['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source']

Rows contributed by each dataset:


,row_count
source,
phishing_email,82486
CEAS_08,39154
Enron,29767
SpamAssasin,5809
Nigerian_Fraud,3332
Ling,2859
Nazario,1565



First five rows:


,subject,body,sender,receiver,date,urls,label,source
0,Never agree to be a loser,"Buck up, your troubles caused by small dimension will soon be over!\nBecome a lover no woman wil...",Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",1,1,CEAS_08
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these techniques http://www.brightmade.com\n\n\n,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",1,1,CEAS_08
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+= >THE DAILY TOP 10 >from CNN.com >T...,Daily Top 10 <Karmandeep-opengevl@universalnet.psi.br>,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",1,1,CEAS_08
3,Re: svn commit: r619753 - in /spamassassin/trunk: lib/Mail/SpamAssassin/PerMsgStatus.pm lib/Mail...,Would anyone object to removing .so from this list? The .so TLD is \nbasically dead and we've ...,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",1,0,CEAS_08
4,SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7iwfna.blu.livefilestore.com/y1pXdX3kwzhBa8xhXv8tdH...,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",1,1,CEAS_08



Combined dataset saved to: C:\Users\nijag\Desktop\Anjaliiiii\data\processed\email\email_combined_before_cleaning.csv


In [ ]:
## Step 9: Identify and Remove Duplicate Rows

**Objective:** Find and remove duplicate records from the combined email dataset.

**Why:** Duplicate records can give repeated emails unnecessary weight and may affect the reliability of model evaluation.

**Method:** Count duplicate rows using `duplicated()`, remove them using `drop_duplicates()`, and compare the row counts before and after cleaning.

**Expected result:** 408 duplicate rows removed, leaving 164,564 rows.


In [9]:

# STEP 9: Identify and remove duplicate rows

rows_before = len(combined_email)
duplicate_count = combined_email.duplicated().sum()

print("Rows before duplicate removal:", rows_before)
print("Duplicate rows found:", duplicate_count)

# Remove exact duplicate rows
combined_email = combined_email.drop_duplicates().reset_index(drop=True)

rows_after = len(combined_email)
rows_removed = rows_before - rows_after

print("\nRows after duplicate removal:", rows_after)
print("Total rows removed:", rows_removed)

# Save the duplicate-removal summary
duplicate_summary = pd.DataFrame({
    "rows_before": [rows_before],
    "duplicate_rows_found": [duplicate_count],
    "rows_after": [rows_after],
    "rows_removed": [rows_removed]
})

duplicate_summary_path = (
    PROCESSED_EMAIL_DIR / "combined_duplicate_summary.csv"
)

duplicate_summary.to_csv(duplicate_summary_path, index=False)

print("\nDuplicate summary saved to:", duplicate_summary_path)

Rows before duplicate removal: 164972
Duplicate rows found: 408

Rows after duplicate removal: 164564
Total rows removed: 408

Duplicate summary saved to: C:\Users\nijag\Desktop\Anjaliiiii\data\processed\email\combined_duplicate_summary.csv


In [ ]:
## Step 10: Create the Combined Email Text

**Objective:** Combine the subject and body of each email into a single `email_text` column.

**Why:** The subject and body may both contain useful clues for distinguishing phishing emails from legitimate emails.

**Method:** Replace missing subject/body values with empty strings, combine both fields, remove leading and trailing spaces, and discard rows where the resulting text is empty.

**Note:** This step prepares the text only. Feature extraction is handled separately in the feature-extraction notebook.


In [10]:

# STEP 10: Create the combined email text column

# Replace missing subject and body values with empty strings
combined_email["subject"] = combined_email["subject"].fillna("")
combined_email["body"] = combined_email["body"].fillna("")

# Combine subject and body into one text column
combined_email["email_text"] = (
    combined_email["subject"].astype(str)
    + " "
    + combined_email["body"].astype(str)
).str.strip()

print("Combined email text column created.")

print("\nExamples of combined email text:")
display(combined_email[["subject", "body", "email_text"]].head(5))

# Identify emails with no usable text
empty_text_count = (
    combined_email["email_text"].str.strip().eq("").sum()
)

print("\nEmails with empty text:", empty_text_count)

# Remove emails that have neither subject nor body text
combined_email = combined_email[
    combined_email["email_text"].str.strip().ne("")
].reset_index(drop=True)

print("Rows after removing empty-text emails:", len(combined_email))
print("Email text preparation completed!")

Combined email text column created.

Examples of combined email text:


,subject,body,email_text
0,Never agree to be a loser,"Buck up, your troubles caused by small dimension will soon be over!\nBecome a lover no woman wil...","Never agree to be a loser Buck up, your troubles caused by small dimension will soon be over!\nB..."
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these techniques http://www.brightmade.com\n\n\n,Befriend Jenna Jameson \nUpgrade your sex and pleasures with these techniques http://www.brightm...
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+= >THE DAILY TOP 10 >from CNN.com >T...,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+= >THE DAILY TO...
3,Re: svn commit: r619753 - in /spamassassin/trunk: lib/Mail/SpamAssassin/PerMsgStatus.pm lib/Mail...,Would anyone object to removing .so from this list? The .so TLD is \nbasically dead and we've ...,Re: svn commit: r619753 - in /spamassassin/trunk: lib/Mail/SpamAssassin/PerMsgStatus.pm lib/Mail...
4,SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7iwfna.blu.livefilestore.com/y1pXdX3kwzhBa8xhXv8tdH...,SpecialPricesPharmMoreinfo \nWelcomeFastShippingCustomerSupport\nhttp://7iwfna.blu.livefilestore...



Emails with empty text: 1
Rows after removing empty-text emails: 164563
Email text preparation completed!


In [ ]:
## Step 11: Validate Missing Values and Labels

**Objective:** Check missing values, label distribution, and empty email text after cleaning.

**Why:** The model needs valid target labels, and the email text must not be empty. Some metadata, such as sender, receiver, date, and URLs, may legitimately be unavailable in the original datasets.

**Important:** Missing metadata is not automatically an error. We should not invent information to fill those fields. We must also verify that the label meanings are consistent across the original datasets before model training.


In [11]:

# STEP 11: Validate the cleaned email data

print("Current dataset shape:", combined_email.shape)

print("\nMissing values in each column:")
display(
    combined_email.isna()
    .sum()
    .to_frame("missing_values")
)

print("\nLabel distribution:")
display(
    combined_email["label"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

print("\nUnique label values:")
print(combined_email["label"].value_counts().index.tolist())

print("\nEmpty email_text rows:",
      combined_email["email_text"].str.strip().eq("").sum())

print("\nDuplicate rows remaining:", combined_email.duplicated().sum())

# Basic validation checks
assert combined_email["label"].notna().all(), "Missing labels found!"
assert combined_email["label"].isin([0, 1]).all(), "Unexpected label values found!"
assert combined_email["email_text"].str.strip().ne("").all(), "Empty email text found!"
assert combined_email.duplicated().sum() == 0, "Duplicate rows remain!"

print("\nAll basic validation checks passed!")

Current dataset shape: (164563, 9)

Missing values in each column:


,missing_values
subject,0
body,0
sender,115034
receiver,116795
date,115186
urls,114703
label,0
source,0
email_text,0



Label distribution:


,count
label,
0,78828
1,85735



Unique label values:
[1, 0]

Empty email_text rows: 0

Duplicate rows remaining: 0

All basic validation checks passed!


In [ ]:
## Step 12: Count URLs and Save the Cleaned Dataset

**Objective:** Count URLs found in each email and save the cleaned dataset.

**Why:** The number of URLs may provide a useful signal for phishing detection. Emails containing many URLs may deserve further investigation, although URL count alone cannot determine whether an email is malicious.

**Method:** Use a regular expression to count text patterns beginning with `http://`, `https://`, or `www.`. Save the cleaned data to `data/processed/email/email_cleaned.csv`.

**Note:** This count is an approximate text-based count. It does not guarantee that every URL is detected or that every detected match is a valid URL.


In [12]:

# STEP 12: Count URLs and save the cleaned dataset

# Pattern for common URL formats
url_pattern = r"https?://\S+|www\.\S+"

# Count URL-like patterns in each email's combined text
combined_email["extracted_url_count"] = (
    combined_email["email_text"]
    .apply(lambda text: len(re.findall(url_pattern, str(text), flags=re.IGNORECASE)))
)

# Arrange columns in a consistent order
final_columns = [
    "subject", "body", "sender", "receiver",
    "date", "urls", "label", "source",
    "extracted_url_count", "email_text"
]

email_cleaned = combined_email[final_columns].copy()

# Save the final cleaned dataset
FINAL_EMAIL_PATH = PROCESSED_EMAIL_DIR / "email_cleaned.csv"
email_cleaned.to_csv(FINAL_EMAIL_PATH, index=False)

print("URL counting completed.")
print("\nFinal dataset shape:", email_cleaned.shape)
print("\nFinal columns:")
print(email_cleaned.columns.tolist())

print("\nURL count statistics:")
display(email_cleaned["extracted_url_count"].describe().to_frame())

print("\nFirst five rows:")
display(
    email_cleaned[
        ["source", "label", "extracted_url_count", "email_text"]
    ].head()
)

print("\nCleaned dataset saved successfully to:")
print(FINAL_EMAIL_PATH)

URL counting completed.

Final dataset shape: (164563, 10)

Final columns:
['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source', 'extracted_url_count', 'email_text']

URL count statistics:


,extracted_url_count
count,164563.000000
mean,0.969781
std,9.421681
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,3134.000000



First five rows:


,source,label,extracted_url_count,email_text
0,CEAS_08,1,1,"Never agree to be a loser Buck up, your troubles caused by small dimension will soon be over!\nB..."
1,CEAS_08,1,1,Befriend Jenna Jameson \nUpgrade your sex and pleasures with these techniques http://www.brightm...
2,CEAS_08,1,24,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+= >THE DAILY TO...
3,CEAS_08,0,652,Re: svn commit: r619753 - in /spamassassin/trunk: lib/Mail/SpamAssassin/PerMsgStatus.pm lib/Mail...
4,CEAS_08,1,1,SpecialPricesPharmMoreinfo \nWelcomeFastShippingCustomerSupport\nhttp://7iwfna.blu.livefilestore...



Cleaned dataset saved successfully to:
C:\Users\nijag\Desktop\Anjaliiiii\data\processed\email\email_cleaned.csv


In [ ]:
## Step 13: Final Verification

**Objective:** Verify the saved `email_cleaned.csv` file.

**Why:** Rechecking the saved file ensures that the cleaning process produced a usable output and that the data can be loaded again for later stages of the project.

**Checks performed:**

* Confirm the output file exists.
* Reload the CSV.
* Verify the expected columns and dataset shape.
* Check labels, missing labels, empty email text, and duplicate rows.
* Confirm the URL count column is present.

**Expected result:** 164,563 rows and 10 columns, with no missing labels, empty email text, or duplicate rows.


In [13]:

# STEP 13: Final verification of the saved dataset

assert FINAL_EMAIL_PATH.exists(), "Cleaned CSV file was not found!"

# Reload the saved file
verified_email = pd.read_csv(FINAL_EMAIL_PATH)

print("File exists:", FINAL_EMAIL_PATH.exists())
print("Verified dataset shape:", verified_email.shape)

print("\nVerified columns:")
print(verified_email.columns.tolist())

print("\nLabel distribution:")
display(
    verified_email["label"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

print("\nMissing labels:", verified_email["label"].isna().sum())
print(
    "Empty email text:",
    verified_email["email_text"].fillna("").str.strip().eq("").sum()
)
print("Duplicate rows:", verified_email.duplicated().sum())
print(
    "URL count column present:",
    "extracted_url_count" in verified_email.columns
)

# Final checks
expected_columns = [
    "subject", "body", "sender", "receiver",
    "date", "urls", "label", "source",
    "extracted_url_count", "email_text"
]

assert verified_email.columns.tolist() == expected_columns
assert verified_email.shape == (164563, 10)
assert verified_email["label"].notna().all()
assert verified_email["label"].isin([0, 1]).all()
assert verified_email["email_text"].fillna("").str.strip().ne("").all()
assert verified_email.duplicated().sum() == 0
assert verified_email["extracted_url_count"].notna().all()

print("\nSUCCESS: All final verification checks passed!")
print("The cleaned dataset is ready for the next stage.")

File exists: True
Verified dataset shape: (164563, 10)

Verified columns:
['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source', 'extracted_url_count', 'email_text']

Label distribution:


,count
label,
0,78828
1,85735



Missing labels: 0
Empty email text: 0
Duplicate rows: 0
URL count column present: True

SUCCESS: All final verification checks passed!
The cleaned dataset is ready for the next stage.


In [14]:

# STEP 1: Check label distribution for every source dataset

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\nijag\Desktop\Anjaliiiii")
CLEANED_FILE = (
    PROJECT_ROOT / "data" / "processed" / "email" / "email_cleaned.csv"
)

email_df = pd.read_csv(CLEANED_FILE)

print("Dataset shape:", email_df.shape)
print("\nLabel distribution by source:")

label_check = pd.crosstab(
    email_df["source"],
    email_df["label"],
    margins=True,
    dropna=False
)

display(label_check)

print("\nLabels present in each source:")
for source, group in email_df.groupby("source"):
    print(f"{source}: {sorted(group['label'].dropna().unique().tolist())}")

Dataset shape: (164563, 10)

Label distribution by source:


label,0,1,All
source,,,
CEAS_08,17312,21842,39154
Enron,15791,13976,29767
Ling,2401,458,2859
Nazario,0,1565,1565
Nigerian_Fraud,0,3332,3332
SpamAssasin,4091,1718,5809
phishing_email,39233,42844,82077
All,78828,85735,164563



Labels present in each source:
CEAS_08: [0, 1]
Enron: [0, 1]
Ling: [0, 1]
Nazario: [1]
Nigerian_Fraud: [1]
SpamAssasin: [0, 1]
phishing_email: [0, 1]


In [15]:

# Check original label distributions before cleaning

for name, df in datasets.items():
    print(f"\nDataset: {name}")
    print("Label counts:")
    print(df["label"].value_counts(dropna=False).sort_index())

    print("Original columns:", df.columns.tolist())


Dataset: CEAS_08
Label counts:
label
0    17312
1    21842
Name: count, dtype: int64
Original columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']

Dataset: Enron
Label counts:
label
0    15791
1    13976
Name: count, dtype: int64
Original columns: ['subject', 'body', 'label']

Dataset: Ling
Label counts:
label
0    2401
1     458
Name: count, dtype: int64
Original columns: ['subject', 'body', 'label']

Dataset: Nazario
Label counts:
label
1    1565
Name: count, dtype: int64
Original columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']

Dataset: Nigerian_Fraud
Label counts:
label
1    3332
Name: count, dtype: int64
Original columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']

Dataset: phishing_email
Label counts:
label
0    39595
1    42891
Name: count, dtype: int64
Original columns: ['text_combined', 'label']

Dataset: SpamAssasin
Label counts:
label
0    4091
1    1718
Name: count, dtype: int64
Original columns: [